In [ ]:
# pandas for data manipulation
# re for regular expressions
import re

# pyplot for plotting
import numpy as np
import pandas as pd

from utils.import_data import get_csv_files, sort_meta_info


In [ ]:
path_to: str = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project"
path: str = path_to + "\\CSI DATA"

# scenario -> location -> user -> esp -> trial -> file_path
FileMap = dict[str, dict[str, dict[str, dict[str, dict[str, str]]]]]
# scenario -> location -> user -> esp -> trial -> csi ndarray
csi_map = dict[str, dict[str, dict[str, dict[str, dict[str, np.ndarray]]]]]


all_data_files = get_csv_files(path)
scenarios_id, locations_id, users_id, esps_id, trials_id = sort_meta_info(path)

print(f"Scenarios present: {', '.join(scenarios_id) or 'none'}")
print(f"Locations present: {', '.join(locations_id) or 'none'}")
print(f"Users present: {', '.join(users_id) or 'none'}")


In [ ]:
def process_csi(data_file: str, its5ghz: bool) -> tuple[np.ndarray, int, int, int]:
    file_csv = pd.read_csv(data_file, header=None)
    # 5 GHz saved CSV header from app_main.c UDP payload: type,seq,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
    # 2.4 GHz saved CSV header from app_main.c UDP payload: type,id,mac,rssi,rate,sig_mode,mcs,bandwidth,smoothing,not_sounding,aggregation,stbc,fec_coding,sgi,noise_floor,ampdu_cnt,channel,secondary_channel,local_timestamp,ant,sig_len,rx_state,len,first_word,data
    acg_gain: float = 0

    # number of samples
    if its5ghz:
        print("\tCom 5 GHz")
        csi_raw: pd.Series = file_csv.iloc[:, 14]
        acg_gain = file_csv.iloc[0, 7]
    else:
        csi_raw: pd.Series = file_csv.iloc[:, 24]

    total_sc_2_4: int = 128
    total_sc_5: int = 106
    valid_csi: list[list[float]] = []

    no_match_count: int = 0
    no_complete_count: int = 0

    for entry in csi_raw:
        match = re.search(r"\[(.*?)\]", str(entry))
        if not match:
            no_match_count += 1
            continue

        nums = [float(n) for n in re.findall(r"-?\d+", match.group(1))]

        if (its5ghz and len(nums) == total_sc_5) or (not its5ghz and len(nums) == total_sc_2_4):
            valid_csi.append(nums)
        else:
            no_complete_count += 1

    valid_csi = np.array(valid_csi)

    print(f"\tTotal CSI entries: {len(csi_raw)}")
    print(f"\tValid CSI entries: {len(valid_csi)}")
    print(f"\tInvalid CSI entries (no match): {no_match_count}")
    print(f"\tInvalid CSI entries (incomplete): {no_complete_count}")
    print(f"\tValid CSI shape: {valid_csi.shape}\n")

    if not its5ghz:
        # (n_amostras, n_subcarriers)
        complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]

        # coloca sc DC no centro (index 32)
        fft_csi = np.fft.fftshift(complex_csi, axes=1)

        # Extract the 52 active subcarriers (IEEE 802.11n standard): [6:58]
        # (n_amostras, 52)
        active_sc = fft_csi[:, 6:58]

        # Remove subcarriers at positions 25, 26, 27 (center)
        # (n_amostras, 49)
        active_sc = np.delete(active_sc, [25, 26, 27], axis=1)

        # seleciona sub_carriers: 2 a 47
        active_sc = active_sc[:, 2:48]
        active_sc = np.abs(active_sc)
    else:
        imag = valid_csi[:, ::2]
        real = valid_csi[:, 1::2]
        complex_csi = real + 1j * imag
        complex_csi = np.delete(complex_csi, [26, 27], axis=1)
        active_sc = np.abs(complex_csi)

    return active_sc, no_match_count, no_complete_count, len(csi_raw)


def process_magnitude(data_files: FileMap) -> csi_map:
    magnitudes = {}
    no_match_count: int = 0
    no_complete_count: int = 0
    total_entries: int = 0

    for scenario_key, locations_map in data_files.items():
        print(f"Processing scenario: {scenario_key}")
        magnitudes[scenario_key] = {}

        for location_key, users_map in locations_map.items():
            magnitudes[scenario_key][location_key] = {}

            for user_key, esps_map in users_map.items():
                magnitudes[scenario_key][location_key][user_key] = {}

                for esp_key, trials_map in esps_map.items():
                    magnitudes[scenario_key][location_key][user_key][esp_key] = {}
                    esp_id = int(esp_key.removeprefix("esp_"))
                    its5ghz = 11 <= esp_id <= 20
                    print("Esp key", esp_key, "5GHz:", its5ghz)

                    for trial_key, file_path in trials_map.items():
                        if file_path is None:
                            continue

                        (
                            magnitudes[scenario_key][location_key][user_key][esp_key][trial_key],
                            no_match,
                            no_complete,
                            total,
                        ) = process_csi(str(file_path), its5ghz)

                        no_match_count += no_match
                        no_complete_count += no_complete
                        total_entries += total

        print(f"Total invalid CSI entries (no match): {no_match_count}")
        print(f"Total invalid CSI entries (incomplete): {no_complete_count}")
        if total_entries > 0:
            print(f"Total percentage of no match: {no_match_count / total_entries:.2%}")
            print(f"Total percentage of incomplete: {no_complete_count / total_entries:.2%}\n\n")
        else:
            print("No entries processed.\n\n")

        no_match_count = 0
        no_complete_count = 0
        total_entries = 0

    return magnitudes

In [ ]:
magnitude_data = process_magnitude(all_data_files)